In [ ]:
# imports
import os
import sys
import torch
from tqdm.notebook import tqdm
sys.path.append("/export/share/peters57dm/Verbund/deepsync/experiments/")
from helper.datasets import (
    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_mnist,
    load_fmnist,
    load_cifar10,
    load_coil20,
    load_coil100,
    load_cifar100,
    load_weizmann,
    load_data,
    data_path,
)
from helper.deep import detect_device
from clustpy.deep.neural_networks.feedforward_autoencoder import FeedforwardAutoencoder
from clustpy.deep import encode_batchwise, get_dataloader

[WARNING] Could not import nltk in clustpy.data.real_world_data. Please install nltk by 'pip install nltk' if necessary


In [3]:
# Parameters
max_embed_size = 10
batch_size = 256
n_epochs = 100 # 50
lr = 1e-3
pretrianed_models_path = "/export/share/peters57dm/Verbund/data/version-2_mse_pretrained_models500-500-2000"
device = detect_device()

os.makedirs(pretrianed_models_path, exist_ok=True)

datasets_loading_methods = [
                            load_pendigits,
                            load_optdigits,
                            load_letterrecognition,
                            load_gaussian_blobs,
                            load_example,
                            load_usps,
                            load_htru,
                            load_har,
                            load_mice,
                            load_synth_high,
                            load_synth_low,
                            load_mnist,
                            load_fmnist,
                            load_cifar10,
                            load_cifar100,
                            load_coil20,
                            load_coil100,
                            load_weizmann,
                            ]

In [4]:
for loading_method in tqdm(datasets_loading_methods, total=len(datasets_loading_methods)):
    data, gt_labels, data_name = load_data(loading_method=loading_method)
    embedded_space_dim = min(data.shape[1], max_embed_size)
    model = FeedforwardAutoencoder(layers=[data.shape[1], 500, 500, 2000, embedded_space_dim]).to(device)
    trainloader = get_dataloader(data, batch_size, shuffle=True)
    _pretrained_model_path = os.path.join(pretrianed_models_path, f'pretrained_{data_name}.pth')
    model.fit(n_epochs=n_epochs, optimizer_params={"lr":lr},
            dataloader=trainloader, model_path=_pretrained_model_path,
            optimizer_class=lambda params, lr: torch.optim.AdamW(params, lr))

  0%|          | 0/18 [00:00<?, ?it/s]

AE training: 100%|██████████| 100/100 [05:27<00:00,  3.28s/it, Training Loss=20.6]


Files already downloaded and verified
Files already downloaded and verified


AE training: 100%|██████████| 100/100 [07:07<00:00,  4.27s/it, Training Loss=61.6]


Files already downloaded and verified
Files already downloaded and verified


AE training: 100%|██████████| 100/100 [10:35<00:00,  6.35s/it, Training Loss=1.1]


In [ ]:
data, gt_labels, data_name = load_data(loading_method=load_gaussian_blobs)
ssl_loss_fn = torch.nn.MSELoss()
embedded_space_dim = min(data.shape[1], max_embed_size)
model = FeedforwardAutoencoder(layers=[data.shape[1], 256, 128, 64, embedded_space_dim]).to(device)
trainloader = get_dataloader(data, batch_size, shuffle=True)
model.fit(n_epochs=n_epochs, optimizer_params={"lr":lr},
            dataloader=trainloader, ssl_loss_fn=ssl_loss_fn,
            optimizer_class=lambda params, lr: torch.optim.AdamW(params, lr))
mse_loss = model.evaluate(trainloader, ssl_loss_fn, device)

AE training:   0%|          | 0/100 [00:00<?, ?it/s]

AE training: 100%|██████████| 100/100 [00:05<00:00, 19.91it/s, Training Loss=0.000111]


FeedforwardAutoencoder(
  (encoder): FullyConnectedBlock(
    (block): Sequential(
      (0): Linear(in_features=2, out_features=256, bias=True)
      (1): LeakyReLU(negative_slope=0.01)
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): LeakyReLU(negative_slope=0.01)
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): LeakyReLU(negative_slope=0.01)
      (6): Linear(in_features=64, out_features=2, bias=True)
    )
  )
  (decoder): FullyConnectedBlock(
    (block): Sequential(
      (0): Linear(in_features=2, out_features=64, bias=True)
      (1): LeakyReLU(negative_slope=0.01)
      (2): Linear(in_features=64, out_features=128, bias=True)
      (3): LeakyReLU(negative_slope=0.01)
      (4): Linear(in_features=128, out_features=256, bias=True)
      (5): LeakyReLU(negative_slope=0.01)
      (6): Linear(in_features=256, out_features=2, bias=True)
    )
  )
)

In [16]:
model.evaluate(trainloader, torch.nn.MSELoss(), device)

tensor(1.2756e-05)